<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Part 5: Vector Differentiation

**Vector differentiation calculates how changes in early and late cooling propagate through temperatures, performance outputs, and the scalar score.**

Part 4 distinguished equal-score contours from feasibility. The classroom system model, performance mapping, and requirement limits remain fixed. This lecture adds derivatives with respect to the decision vector: gradients for scalar outputs, Jacobians for several outputs, and Hessians for curvature. These calculations supply the local information used in gradient descent.

### 1 · Classroom formulation

The real decision is how much cooling to use early and late. More cooling can reduce heat discomfort, but it uses energy and may make the room too cold.

Let \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). The horizon has \\(n=12\\) decision steps, with actions \\(u_0,\ldots,u_{11}\\) and states \\(T_0,\ldots,T_{12}\\).

> $\displaystyle u_t=\begin{cases}u_{\mathrm{early}},&t=0,\ldots,5,\\u_{\mathrm{late}},&t=6,\ldots,11.\end{cases}$

Indoor temperature \\(T_t\\) is produced by the system; it is not chosen directly. With initial temperature \\(T_0=27\,^{\circ}\mathrm C\\), the physical transition is

> $\displaystyle T_{t+1}=F(T_t,T_t^{\mathrm{out}},N_t,u_t;a,b,c)$
>
> $\displaystyle \phantom{T_{t+1}}=T_t+a(T_t^{\mathrm{out}}-T_t)+bN_t-cu_t,\quad t=0,\ldots,11.$

| Role | Values held fixed in the demonstrations |
|:---|:---|
| Fixed parameters | $(a,b,c)=(0.12,0.012,0.45)$ |
| External inputs | $T_t^{\mathrm{out}}=31\,^{\circ}\mathrm C$ and $N_t=20$ people at every step |
| Cooling limits | $u_{\min}=0$, $u_{\max}=5$ cooling units |
| State limits | $T_{\min}=20\,^{\circ}\mathrm C$, $T_{\max}=30\,^{\circ}\mathrm C$ |
| Energy limit | $E_{\max}=60$ model energy units |
| Energy-weight hyperparameter | $\lambda_E=1$ unless stated otherwise |

The performance mapping \\(G\\) measures discomfort outside the 22–24 °C comfort range and energy use:

> $\displaystyle D(u)=\sum_{t=1}^{12}\left[\max(T_t-24,0)^2+\max(22-T_t,0)^2\right].$
>
> $\displaystyle E(u)=\frac12\sum_{t=0}^{11}u_t^2=3\left(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2\right).$

\\(D\\) uses squared-temperature step units. \\(E\\) uses model energy units, not calibrated kWh. The weight converts energy into the chosen score scale:

> $\displaystyle J(u;\lambda_E)=H(D(u),E(u);\lambda_E)=D(u)+\lambda_EE(u).$

Feasibility requires cooling bounds, all state limits for \\(t=1,\ldots,12\\), and \\(E(u)\le E_{\max}\\). The comfort range is a performance target; the wider 20–30 °C range is a hard requirement.

In standard notation, \\(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\\) and \\(y=\operatorname{Sim}(x)\\). Here \\(\operatorname{Sim}\\) composes repeated \\(F\\) transitions with \\(G\\). The objective \\(f(y;\lambda_E)\\) is the standard-form name for the score supplied by \\(H\\). The score depends on the cooling decision through both the simulated states and energy use.

### 2 · Partial derivatives, gradients, and differentials

Increasing early cooling while holding late cooling fixed changes both the temperature path and the score. A **partial derivative** measures the score change per small change in that one decision coordinate. Let \\(e_1=(1,0)\\) and \\(e_2=(0,1)\\) be the coordinate directions. At fixed \\(\lambda_E\\),

> $\displaystyle \frac{\partial J}{\partial u_i}(u;\lambda_E)=\lim_{\varepsilon\to0}\frac{J(u+\varepsilon e_i;\lambda_E)-J(u;\lambda_E)}{\varepsilon},\quad i=1,2.$

Here \\(u_1=u_{\mathrm{early}}\\), \\(u_2=u_{\mathrm{late}}\\), and \\(\varepsilon\\) is a signed cooling perturbation. The **gradient** collects these scalar slopes into a column, in decision-coordinate order:

> $\displaystyle \nabla_u J=\begin{bmatrix}\partial J/\partial u_{\mathrm{early}}\\\partial J/\partial u_{\mathrm{late}}\end{bmatrix}\in\mathbb R^{2\times1}.$

For a differentiable score, the differential is the linear part of its change:

> $\displaystyle \mathrm dJ=\nabla_u J^{\mathsf T}\,\mathrm du.$
>
> $\displaystyle J(u+\Delta u;\lambda_E)-J(u;\lambda_E)=\nabla_u J^{\mathsf T}\Delta u+o(\lVert\Delta u\rVert_2).$

The differential \\(\mathrm dJ\\) is a scalar; the gradient is a vector. The remainder divided by \\(\lVert\Delta u\rVert_2\\) tends to zero as the displacement tends to zero. Thus the inner product predicts the leading score change, not generally the exact finite change. Existence of partial derivatives alone does not guarantee differentiability for an arbitrary function.

For energy, \\(E(u)=3(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2)\\), so \\(\nabla_u E=6u\\). At \\(u=(3,2)\\), \\(\nabla_u E=(18,12)^{\mathsf T}\\). An early-cooling increase of 0.01 predicts an energy increase of 0.18; the exact increase is 0.1803 model energy units. The small difference is a quadratic term.

### 3 · Linear and quadratic differentiation rules

The energy calculation is an instance of a general vector rule. In this section \\(x\in\mathbb R^p\\) is a column decision vector, \\(v\in\mathbb R^p\\), \\(A\in\mathbb R^{q\times p}\\), \\(z\in\mathbb R^q\\), and \\(Q\in\mathbb R^{p\times p}\\). All quantities except \\(x\\) are held fixed. A gradient has \\(p\\) components even when its scalar expression contains matrix products.

| Scalar expression | Gradient with respect to $x$ |
|:---|:---|
| $v^{\mathsf T}x$ | $v$ |
| $x^{\mathsf T}x=\lVert x\rVert_2^2$ | $2x$ |
| $x^{\mathsf T}Qx$ | $(Q+Q^{\mathsf T})x$ |
| $\frac12\lVert Ax-z\rVert_2^2$ | $A^{\mathsf T}(Ax-z)$ |

The quadratic rule follows by collecting the coefficients of \\(\mathrm dx\\):

> $\displaystyle \mathrm d(x^{\mathsf T}Qx)=(\mathrm dx)^{\mathsf T}Qx+x^{\mathsf T}Q\,\mathrm dx$
>
> $\displaystyle \phantom{\mathrm d(x^{\mathsf T}Qx)}=\big[(Q+Q^{\mathsf T})x\big]^{\mathsf T}\mathrm dx.$

Only when \\(Q=Q^{\mathsf T}\\) does this simplify to \\(2Qx\\). In the classroom, \\(x=u\\), \\(Q=3I_2\\), and \\(I_2\\) is the two-dimensional identity matrix. The rule gives \\(\nabla_u E=6u\\).

For the squared-residual rule, write \\(r=Ax-z\\). Since \\(\mathrm dr=A\,\mathrm dx\\),

> $\displaystyle \mathrm d\!\left(\tfrac12 r^{\mathsf T}r\right)=r^{\mathsf T}A\,\mathrm dx=\big[A^{\mathsf T}(Ax-z)\big]^{\mathsf T}\mathrm dx.$

This form will also describe classroom discomfort within a region where the same temperatures lie above, inside, or below the comfort range. The squared norm is differentiable everywhere. The unsquared Euclidean norm has gradient \\(x/\lVert x\rVert_2\\) only for \\(x\ne0\\); the two expressions must not be interchanged.

### 4 · Jacobians collect derivatives of several outputs

The classroom produces both discomfort and energy. In standard notation, let \\(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\\) and take the response vector to be \\(y=\operatorname{Sim}(x)=[D(u),E(u)]^{\mathsf T}\\). The simulation composes repeated state transitions \\(F\\) with the performance mapping \\(G\\).

For \\(p\\) decision coordinates and \\(q\\) responses, define the **Jacobian** \\(\mathcal A(x)\\) by

> $\displaystyle \mathcal A_{ri}(x)=\frac{\partial y_r}{\partial x_i},\quad r=1,\ldots,q,\quad i=1,\ldots,p.$
>
> $\displaystyle \mathrm dy=\mathcal A(x)\,\mathrm dx,\qquad \mathcal A(x)\in\mathbb R^{q\times p}.$

Rows identify outputs; columns identify decision coordinates. In the classroom \\(p=q=2\\):

> $\displaystyle \mathcal A(u)=\begin{bmatrix}\partial D/\partial u_{\mathrm{early}}&\partial D/\partial u_{\mathrm{late}}\\\partial E/\partial u_{\mathrm{early}}&\partial E/\partial u_{\mathrm{late}}\end{bmatrix}=\begin{bmatrix}(\nabla_u D)^{\mathsf T}\\(\nabla_u E)^{\mathsf T}\end{bmatrix}.$

For example, \\(\mathcal A\Delta u\\) predicts two performance changes. It is not a scalar score change until the objective supplies a comparison rule. The symbol \\(J\\) continues to denote the classroom score, not the Jacobian.

| Mapping | Derivative representation | Shape |
|:---|:---|:---|
| Scalar parameter $s$ to vector $u(s)$ | $\mathrm du/\mathrm ds$ | $p\times1$ |
| Vector $x$ to scalar score | Column gradient | $p\times1$ |
| Vector $x$ to vector $y$ | Jacobian, outputs by inputs | $q\times p$ |

Under this convention, the Jacobian of a scalar-valued function is the row given by its gradient transpose. In NumPy, a gradient is stored as shape <code>(p,)</code> and a Jacobian as <code>(q, p)</code>; <code>jacobian @ displacement</code> produces the response-change vector. Transposing a one-dimensional array does not create a column.

### 5 · The chain rule through the classroom simulation

The standard-form score \\(f(y;\lambda_E)=y_1+\lambda_E y_2\\) is supplied by \\(H\\). Its response gradient is \\(\nabla_y f=(1,\lambda_E)^{\mathsf T}\\). The differential factors through the response mapping:

> $\displaystyle \mathrm dJ=\nabla_y f^{\mathsf T}\mathrm dy=\nabla_y f^{\mathsf T}\mathcal A\,\mathrm dx.$
>
> $\displaystyle \nabla_x\big[f(\operatorname{Sim}(x);\lambda_E)\big]=\mathcal A(x)^{\mathsf T}\nabla_y f.$

The dimensions explain the transpose: \\((p\times q)(q\times1)=p\times1\\). For cooling decisions this gives

> $\displaystyle \nabla_u J(u;\lambda_E)=\nabla_u D(u)+\lambda_E\nabla_u E(u).$

The score gradient includes every path from the decision to performance. In particular, \\(\partial D/\partial T_t\\) is a state sensitivity, not the derivative with respect to a chosen cooling level.

To calculate the missing decision sensitivities, let \\(s_t=\nabla_u T_t\in\mathbb R^{2\times1}\\), and define \\(\eta_t=e_1\\) for \\(t=0,\ldots,5\\) and \\(\eta_t=e_2\\) for \\(t=6,\ldots,11\\). The vector \\(\eta_t\\) selects the cooling period. Differentiating the existing physical transition gives

> $\displaystyle s_0=\begin{bmatrix}0\\0\end{bmatrix},\qquad s_{t+1}=(1-a)s_t-c\eta_t,\quad t=0,\ldots,11.$

The initial state and external inputs are held fixed, so their derivatives with respect to the decision are zero. Early cooling influences both early and later temperatures through the carried-forward state. Late cooling cannot change \\(T_1,\ldots,T_6\\).

Define \\(\theta(u)=[T_1(u),\ldots,T_{12}(u)]^{\mathsf T}\\). Its Jacobian \\(S\in\mathbb R^{12\times2}\\) has row \\(t\\) equal to \\(s_t^{\mathsf T}\\). Since the state transition is affine and its coefficients are fixed, \\(S\\) is independent of \\(u\\), and \\(\theta(u+\Delta u)-\theta(u)=S\Delta u\\) is exact here.

For the discomfort penalty \\(\phi(T)=\max(T-24,0)^2+\max(22-T,0)^2\\),

> $\displaystyle \phi'(T)=2\max(T-24,0)-2\max(22-T,0).$
>
> $\displaystyle w=\begin{bmatrix}\phi'(T_1)\\\vdots\\\phi'(T_{12})\end{bmatrix},\qquad \nabla_u D=S^{\mathsf T}w.$

Both one-sided derivatives are zero at 22 and 24 °C, so \\(\phi\\) is continuously differentiable. The chain rule now yields \\(\nabla_u J=S^{\mathsf T}w+6\lambda_Eu\\). Feasibility remains a separate requirement check after evaluating the state path and performance.

### 6 · Hessians and local curvature

The gradient gives a first-order slope. The **Hessian** differentiates that gradient again. For a scalar-valued function \\(\psi(x)\\),

> $\displaystyle [\nabla_x^2\psi(x)]_{ij}=\frac{\partial}{\partial x_j}\left(\frac{\partial\psi}{\partial x_i}\right),\quad i,j=1,\ldots,p.$

It has shape \\(p\times p\\), and is symmetric where second derivatives are continuous. The diagonal entries measure curvature along individual coordinates; off-diagonal entries measure how one coordinate changes another coordinate's slope. We use \\(\nabla_u^2J\\) for the score Hessian, retaining \\(H\\) for the score mapping.

Within a region where no temperature crosses 22 or 24 °C, define the diagonal matrix \\(W\in\mathbb R^{12\times12}\\) with \\(W_{tt}=1\\) for temperatures outside the comfort range and zero for those inside. For the penalized temperatures, let \\(z_t\\) be 24 °C above the band and 22 °C below it; set \\(z_t=0\\) inside, where the weight is zero. In that region,

> $\displaystyle D(u)=\big[\theta(u)-z\big]^{\mathsf T}W\big[\theta(u)-z\big].$

Because \\(\theta(u)\\) is affine, differentiating the squared-residual form gives

> $\displaystyle \nabla_u^2D=2S^{\mathsf T}WS,\qquad \nabla_u^2E=6I_2.$
>
> $\displaystyle \nabla_u^2J=2S^{\mathsf T}WS+6\lambda_EI_2.$

The second-order prediction adds the curvature correction:

> $\displaystyle J(u+\Delta u;\lambda_E)\approx J(u;\lambda_E)+\nabla_u J^{\mathsf T}\Delta u$
>
> $\displaystyle \phantom{J(u+\Delta u;\lambda_E)\approx}\ +\tfrac12\Delta u^{\mathsf T}\nabla_u^2J\,\Delta u.$

For this piecewise quadratic score, the formula is exact while the displacement stays in the same comfort region. At a comfort threshold the gradient still exists, but the classical Hessian generally does not. The formula above is a regional curvature formula, not a Hessian at the threshold.

For a general twice continuously differentiable objective, a zero gradient and positive-definite Hessian at an interior point establish a strict local minimum. A zero gradient alone does not. Constrained minima require feasibility and boundary analysis, which cannot be replaced by a curvature calculation.

### 7 · Analytical derivatives and numerical verification

The calculation fixes \\(u=(3,2)\\) and \\(\lambda_E=1\\). Simulation, performance calculation, and feasibility checks retain the same definitions as the preceding lecture; sensitivity propagation is added separately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0

# Evaluation hyperparameter
ENERGY_WEIGHT = 1.0

# Stable visual roles
BLUE, ORANGE, GRAY = "#2563EB", "#E88726", "#9CA3AF"

In [ ]:
def expand_decision(decision):
    """Expand the chosen levels into u_0, ..., u_11."""
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("A decision must contain two finite cooling levels.")
    return np.repeat(decision, TIME_STEPS // 2)


def simulate_classroom(decision):
    cooling_schedule = expand_decision(decision)
    temperatures = np.empty(TIME_STEPS + 1)
    temperatures[0] = INITIAL_TEMPERATURE
    for t in range(TIME_STEPS):
        temperatures[t + 1] = (
            temperatures[t]
            + WEATHER_EXCHANGE * (OUTSIDE_TEMPERATURE[t] - temperatures[t])
            + OCCUPANT_HEAT * OCCUPANTS[t]
            - COOLING_EFFECT * cooling_schedule[t]
        )
    return cooling_schedule, temperatures


def performance_outputs(cooling_schedule, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def check_feasibility(decision, temperatures, energy):
    violations = []
    if not np.all((MIN_COOLING <= decision) & (decision <= MAX_COOLING)):
        violations.append("cooling bound")
    if np.min(temperatures[1:]) < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if np.max(temperatures[1:]) > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")
    return tuple(violations)


def evaluate_candidate(decision, energy_weight=ENERGY_WEIGHT):
    decision = np.asarray(decision, dtype=float)
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)
    violations = check_feasibility(decision, temperatures, energy)
    return {
        "decision": decision.copy(), "cooling_schedule": cooling_schedule,
        "temperatures": temperatures, "discomfort": discomfort, "energy": energy,
        "feasible": not violations, "violations": violations,
        "energy_weight": float(energy_weight),
        "objective": discomfort + energy_weight * energy,
    }


def score(decision, energy_weight=ENERGY_WEIGHT):
    """Also defined outside the feasible set for geometry and derivatives."""
    return evaluate_candidate(decision, energy_weight)["objective"]

The derivative function returns the state Jacobian, performance Jacobian, score gradient, and score Hessian with the same evaluated candidate. At a comfort threshold it returns no classical Hessian. A numerical tolerance identifies points too close to the threshold for this curvature demonstration.

In [ ]:
def evaluate_derivatives(decision, energy_weight=ENERGY_WEIGHT):
    result = evaluate_candidate(decision, energy_weight)

    # Differentiate the physical state transition; row zero is the fixed T_0.
    temperature_jacobian = np.zeros((TIME_STEPS + 1, 2))
    for t in range(TIME_STEPS):
        temperature_jacobian[t + 1] = (
            (1 - WEATHER_EXCHANGE) * temperature_jacobian[t]
        )
        period = t // (TIME_STEPS // 2)
        temperature_jacobian[t + 1, period] -= COOLING_EFFECT
    state_jacobian = temperature_jacobian[1:]

    # Differentiate the performance mapping.
    temperatures = result["temperatures"][1:]
    discomfort_slopes = (
        2 * np.maximum(temperatures - 24.0, 0.0)
        - 2 * np.maximum(22.0 - temperatures, 0.0)
    )
    discomfort_gradient = state_jacobian.T @ discomfort_slopes
    energy_gradient = 6 * result["decision"]
    performance_jacobian = np.vstack((discomfort_gradient, energy_gradient))

    # Differentiate the score mapping at a fixed energy weight.
    response_gradient = np.array([1.0, energy_weight])
    score_gradient = performance_jacobian.T @ response_gradient
    near_threshold = np.any(
        np.isclose(temperatures[:, None], [22.0, 24.0], rtol=0, atol=1e-10)
    )
    score_hessian = None
    if not near_threshold:
        penalized = (temperatures < 22.0) | (temperatures > 24.0)
        score_hessian = (
            2 * state_jacobian.T @ (penalized[:, None] * state_jacobian)
            + 6 * energy_weight * np.eye(2)
        )

    return {
        **result,
        "temperature_jacobian": temperature_jacobian,
        "performance_jacobian": performance_jacobian,
        "score_gradient": score_gradient,
        "score_hessian": score_hessian,
    }


def finite_difference_jacobian(mapping, decision, step=1e-4):
    """Central differences: one column per perturbed input coordinate."""
    if not np.isfinite(step) or step <= 0:
        raise ValueError("The finite-difference step must be finite and positive.")
    decision = np.asarray(decision, dtype=float)
    columns = [
        (np.atleast_1d(mapping(decision + step * direction))
         - np.atleast_1d(mapping(decision - step * direction))) / (2 * step)
        for direction in np.eye(decision.size)
    ]
    return np.column_stack(columns)


def performance_vector(decision):
    result = evaluate_candidate(decision)
    return np.array([result["discomfort"], result["energy"]])

At the fixed decision, the performance Jacobian separates physical performance sensitivities from the objective's weighting:

> $\displaystyle \mathcal A(3,2)\approx\begin{bmatrix}-23.152&-11.158\\18&12\end{bmatrix},\qquad \nabla_u J(3,2;1)\approx\begin{bmatrix}-5.152\\0.842\end{bmatrix}.$

More early cooling reduces discomfort at a rate that outweighs its energy cost under \\(\lambda_E=1\\). The late-cooling marginal energy cost slightly exceeds its marginal discomfort reduction. The Jacobian's rows have their respective performance units per cooling unit; the score gradient has score units per cooling unit.

Central finite differences perturb one coordinate at a time and recompute the simulation. Agreement with the analytical derivatives checks the chain-rule calculation at this candidate; it does not establish an optimum. The step \\(\varepsilon>0\\) is a numerical hyperparameter: large values can cross comfort regions, and extremely small values can amplify floating-point error. Infeasible perturbations may define the mathematical derivative but cannot be accepted as real-system decisions.

In [ ]:
decision = np.array([3.0, 2.0])
derivatives = evaluate_derivatives(decision)
print("Feasibility:", "feasible" if derivatives["feasible"] else derivatives["violations"])
print("Performance Jacobian (rows D, E; columns early, late):")
print(np.round(derivatives["performance_jacobian"], 6))
print("Score gradient:", np.round(derivatives["score_gradient"], 6))
print("Score Hessian:\n", np.round(derivatives["score_hessian"], 6))
for perturbation in [1e-2, 1e-3, 1e-4]:
    numerical_jacobian = finite_difference_jacobian(
        performance_vector, decision, step=perturbation
    )
    numerical_gradient = finite_difference_jacobian(
        score, decision, step=perturbation
    )[0]
    numerical_hessian = finite_difference_jacobian(
        lambda trial: evaluate_derivatives(trial)["score_gradient"],
        decision, step=perturbation,
    )
    print(
        f"epsilon={perturbation:g}: max absolute differences "
        f"Jacobian={np.max(np.abs(numerical_jacobian - derivatives['performance_jacobian'])):.2e}, "
        f"gradient={np.max(np.abs(numerical_gradient - derivatives['score_gradient'])):.2e}, "
        f"Hessian={np.max(np.abs(numerical_hessian - derivatives['score_hessian'])):.2e}"
    )

Changing only \\(\lambda_E\\) leaves the state path, performance Jacobian, and feasibility unchanged. It changes how the two Jacobian rows combine into the score gradient. The following rows evaluate the same physical decision; they do not compare competing decisions under different objectives.

In [ ]:
for energy_weight in [0.5, 1.0, 2.0]:
    weighted = evaluate_derivatives(decision, energy_weight)
    print(
        f"lambda_E={energy_weight:.1f}: feasible={weighted['feasible']}; "
        f"D={weighted['discomfort']:.3f}, E={weighted['energy']:.3f}; "
        f"gradient={np.round(weighted['score_gradient'], 3)}"
    )

### 8 · State sensitivity and score curvature

The left panel shows the two columns of the temperature Jacobian across the horizon. Late cooling has zero influence through \\(T_6\\), while early cooling continues to influence the later states. Both curves describe the same fixed physical system.

The right panel holds late cooling at 2 and changes only early cooling around 3, keeping \\(\lambda_E=1\\). It compares the simulated score change with first- and second-order predictions at the same reference decision. The quadratic prediction coincides with simulation until \\(T_6\\) first reaches 24 °C, at an early-cooling increase of about 0.094. Beyond that point the comfort-region curvature changes. All candidates in the displayed interval satisfy the requirement limits.

In [ ]:
def show_vector_differentiation():
    reference = evaluate_derivatives((3.0, 2.0))
    direction = np.array([1.0, 0.0])
    offsets = np.linspace(-0.2, 0.3, 251)
    candidates = [
        evaluate_candidate(reference["decision"] + offset * direction)
        for offset in offsets
    ]
    changes = np.array([r["objective"] - reference["objective"] for r in candidates])
    feasible = np.array([r["feasible"] for r in candidates])
    linear = offsets * (reference["score_gradient"] @ direction)
    quadratic = linear + 0.5 * offsets ** 2 * (
        direction @ reference["score_hessian"] @ direction
    )
    first_crossing = (
        (24.0 - reference["temperatures"][6])
        / reference["temperature_jacobian"][6, 0]
    )

    figure, axes = plt.subplots(1, 2, figsize=(11.8, 4.8))
    for index, label, linestyle in [
        (0, "Sensitivity to early cooling", "-"),
        (1, "Sensitivity to late cooling", "--"),
    ]:
        axes[0].plot(
            np.arange(TIME_STEPS + 1), reference["temperature_jacobian"][:, index],
            color=BLUE, linestyle=linestyle, linewidth=2.3, label=label,
        )
    axes[0].axvline(6, color=GRAY, linestyle=":")
    axes[0].set(
        xlabel="State index t", ylabel="Temperature sensitivity (°C / cooling unit)",
        title="Early cooling also changes later states", xticks=np.arange(0, 13, 2),
    )
    axes[0].legend(loc="lower left", fontsize=8)

    axes[1].plot(offsets, np.where(feasible, changes, np.nan), color=BLUE,
                 linewidth=3, label="Simulated feasible score change")
    if not feasible.all():
        axes[1].plot(offsets, np.where(~feasible, changes, np.nan), color=GRAY,
                     linewidth=3, label="Infeasible score extension")
    axes[1].plot(offsets, linear, "--", color=ORANGE, linewidth=2,
                 label="First-order prediction")
    axes[1].plot(offsets, quadratic, ":", color="#172033", linewidth=2,
                 label="Second-order prediction")
    axes[1].axvline(first_crossing, color=GRAY, linestyle="-.",
                    label="First comfort-region change")
    axes[1].scatter(0, 0, color=ORANGE, s=45, zorder=5)
    axes[1].set(
        xlabel="Early-cooling change (cooling units)",
        ylabel="Score change (score units)",
        title="Curvature improves the local score prediction",
    )
    axes[1].legend(loc="upper right", fontsize=8)
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.spines[["top", "right"]].set_visible(False)
    figure.tight_layout()
    return figure

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/08_vector_differentiation.svg" alt="Temperature sensitivities to early and late cooling, and simulated score changes compared with linear and quadratic derivative predictions" width="940" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
derivative_figure = show_vector_differentiation()
plt.show()
plt.close(derivative_figure)